In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None, "display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
from src import config as C

# 08 — Model Değerlendirme, Eşik, Önem ve Kayıt

Bu notebook'ta sırayla: **(1)** eşiği eğitim verisinde seç, **(2)** test kümesini **bir kez** aç, **(3)** metrikleri belirsizlikle raporla, **(4)** hataları ve önemleri incele, **(5)** modeli kaydet/yükle/doğrula.

## 8.1 Model, bölme ve out-of-fold (OOF) olasılıklar
Eşik, eğitim verisinin **out-of-fold** tahminleriyle seçilir: her müşterinin skoru, o müşteriyi görmemiş bir modelden gelir → eşik seçimi dürüst olur.

In [2]:
import json, joblib
from src import evaluation as E
from src import modeling as M
from src.train import smooth_best_threshold

df = M.load_clean(); train, test = M.split_data(df)
X_tr, y_tr = M.xy(train); X_te, y_te = M.xy(test)
model = joblib.load(C.MODEL_PATH); meta = json.loads(C.MODEL_META_PATH.read_text(encoding="utf-8")); saved = json.loads(C.METRICS_PATH.read_text(encoding="utf-8"))
print("Kayıtlı model:", meta["model"], meta["hyperparameters"], "| özellik seti:", meta["feature_set"])

oof = M.oof_probabilities(model, X_tr, y_tr)    # model unfitted kopyalanır, fold'larda eğitilir

Kayıtlı model: Logistic Regression {'C': 3} | özellik seti: core


## 8.2 Eşik analizi — neden 0,50 değil?
Sınıflandırıcı olasılık üretir; "ayrılacak" demek için bir **eşik** seçilir. Varsayılan 0,50 burada işe yaramaz: hiçbir müşteri 0,5'e yakın skor almıyor, en yüksek skor ≈ 0,6 ve müşterilerin çoğu 0,15'in altında (taban oran %8!). Eşiği **iş değerine** göre seçiyoruz:

$$\text{net değer} = \sum_{\text{aranan, ayrılacak}} (\text{başarı oranı} \times \text{12 ay} \times \text{ücret} \times \text{marj}) - \text{maliyet} \times \text{aranan sayısı}$$

In [3]:
tt = E.threshold_table(y_tr, oof, train["monthly_fee_tl"])
thr = smooth_best_threshold(tt)
display(Markdown(f"Seçilen eşik: **{thr:.2f}** (kayıtlı model eşiği: {meta['threshold']:.2f})"))
assert abs(thr - meta["threshold"]) < 1e-9
display(tt.loc[tt.threshold.isin([0.05, 0.07, 0.09, 0.11, 0.15, 0.20, 0.30]), ["threshold", "flagged_pct", "precision", "recall", "f1", "f2", "net_value_tl"]])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(tt.threshold, tt.precision, label="Precision"); ax[0].plot(tt.threshold, tt.recall, label="Recall"); ax[0].plot(tt.threshold, tt.f2, label="F2")
ax[0].axvline(thr, color="k", ls="--"); ax[0].legend(); ax[0].set_xlabel("Eşik"); ax[0].set_title("Eşik trade-off (train OOF)")
ax[1].plot(tt.threshold, tt.net_value_tl / 1000); ax[1].axvline(thr, color="k", ls="--"); ax[1].axhline(0, color="gray")
ax[1].set_xlabel("Eşik"); ax[1].set_ylabel("Net değer (bin TL)"); ax[1].set_title("Kampanya net değeri (varsayımlarla)")
plt.tight_layout(); plt.show()

Seçilen eşik: **0.10** (kayıtlı model eşiği: 0.10)

,threshold,flagged_pct,precision,recall,f1,f2,net_value_tl
3,0.050,0.540,0.130,0.856,0.226,0.404,"11,440.560"
5,0.070,0.406,0.146,0.720,0.242,0.402,"13,098.416"
7,0.090,0.318,0.166,0.644,0.264,0.409,"14,708.768"
9,0.110,0.242,0.181,0.534,0.270,0.384,"13,639.096"
13,0.150,0.142,0.230,0.398,0.292,0.348,"13,253.064"
18,0.200,0.080,0.296,0.288,0.292,0.290,"11,351.584"
28,0.300,0.028,0.341,0.119,0.176,0.136,"5,152.448"


**Precision–Recall takası:** Eşik düştükçe daha çok ayrılanı yakalarız (Recall ↑) ama daha çok *gereksiz* arama yaparız (Precision ↓). **F2**, Recall'a Precision'ın 2 katı ağırlık verir (kaçırmak daha pahalıysa). Hangi noktanın doğru olduğu **iş maliyetine** bağlı → net değer eğrisi.

### Duyarlılık analizi: sonuç maliyet varsayımına çok bağlı!

In [4]:
sens = pd.read_csv(C.ROOT / "reports" / "cost_sensitivity.csv")
display(sens.rename(columns={"offer_cost_tl": "kampanya_maliyeti_TL", "test_net_value_if_contact_everyone_tl": "herkesi_ararsak_net_TL"}).round(3))

,kampanya_maliyeti_TL,threshold,train_oof_flagged_pct,test_flagged_pct,test_recall,test_precision,test_net_value_tl,herkesi_ararsak_net_TL
0,20,0.030,0.728,0.697,0.900,0.108,"7,730.912","7,008.480"
1,50,0.100,0.276,0.306,0.600,0.164,"2,961.512","-3,791.520"
2,100,0.200,0.080,0.092,0.233,0.212,-141.144,"-21,791.520"
3,150,0.370,0.019,0.017,0.000,0.000,-900.000,"-39,791.520"


**Okuma:**
* **Düşük maliyet (20–50 TL):** Model değer katıyor; 50 TL'de test'te net değer pozitif, **herkesi aramak ise zararlı**. 20 TL'de herkesi aramak neredeyse aynı kadar iyi → model neredeyse gereksiz.
* **Yüksek maliyet (100–150 TL):** Model bu kampanyayı kârlı yapacak kadar keskin değil; çok az müşteri eşiği geçiyor ve net değer sıfıra yakın/negatif.
* **Ders:** "Model iyi mi?" sorusunun cevabı, *"hangi aksiyon için?"* sorusuna bağlıdır. Bu yüzden gerçek maliyetler iş birimiyle konuşulmalı. Varsayılan senaryomuz 50 TL.

## 8.3 Final: test kümesini AÇIYORUZ (tek seferlik)

In [5]:
p_te = model.predict_proba(X_te)[:, 1]
rank = E.ranking_metrics(y_te, p_te); at = E.threshold_metrics(y_te, p_te, thr)
at["net_value_tl"] = E.net_value(y_te, p_te >= thr, test["monthly_fee_tl"])
display(pd.Series({**rank, **{k: v for k, v in at.items() if k in ["flagged", "flagged_pct", "precision", "recall", "f1", "f2", "net_value_tl"]}}, name="TEST").to_frame())
# Kayıtlı metrics.json ile birebir aynı mı?
assert abs(rank["pr_auc"] - saved["test_ranking"]["pr_auc"]) < 1e-12 and abs(at["recall"] - saved["test_at_threshold"]["recall"]) < 1e-12
print("✔ Notebook sonuçları kayıtlı metrics.json ile birebir aynı.")

,TEST
roc_auc,0.715
pr_auc,0.164
brier,0.075
flagged,110.000
flagged_pct,0.306
precision,0.164
recall,0.600
f1,0.257
f2,0.391
net_value_tl,"2,961.512"


✔ Notebook sonuçları kayıtlı metrics.json ile birebir aynı.


In [6]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, RocCurveDisplay, PrecisionRecallDisplay
pred = (p_te >= thr).astype(int)
print(classification_report(y_te, pred, target_names=["Kalır", "Ayrılır"], zero_division=0))
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=["Kalır", "Ayrılır"], cmap="Blues", ax=ax[0], colorbar=False); ax[0].grid(False); ax[0].set_title(f"Karışıklık matrisi (eşik {thr:.2f})")
RocCurveDisplay.from_predictions(y_te, p_te, ax=ax[1]); ax[1].plot([0, 1], [0, 1], "k--"); ax[1].set_title("ROC")
PrecisionRecallDisplay.from_predictions(y_te, p_te, ax=ax[2]); ax[2].axhline(y_te.mean(), color="k", ls="--"); ax[2].set_title("Precision-Recall")
plt.tight_layout(); plt.show()

              precision    recall  f1-score   support

       Kalır       0.95      0.72      0.82       330
     Ayrılır       0.16      0.60      0.26        30

    accuracy                           0.71       360
   macro avg       0.56      0.66      0.54       360
weighted avg       0.89      0.71      0.77       360



### Karışıklık matrisinin iş anlamı
* **TN:** Kalacak müşteriyi doğru bildik. **FP:** Kalacak müşteriyi "riskli" dedik (gereksiz arama maliyeti).
* **FN:** Ayrılacak müşteriyi kaçırdık (en pahalı hata olabilir). **TP:** Ayrılacak müşteriyi yakaladık.

## 8.4 Küçük test kümesi: sonuçlar ne kadar güvenilir? (Bootstrap %95 aralık)
Testte yalnızca **~30 ayrılan** var; tek bir müşteri bile metrikleri oynatır. Bu yüzden nokta tahminin yanına belirsizlik koymak zorundayız.

In [7]:
ci = E.bootstrap_ci(y_te, p_te, thr); display(ci.round(3))
base = saved["test_baseline_dummy"]
display(Markdown(f"Baseline (dummy) test ROC-AUC = {base['roc_auc']:.2f}, PR-AUC = {base['pr_auc']:.3f} (taban oran). Model PR-AUC **{rank['pr_auc']:.3f}** → taban orana göre ≈ **{rank['pr_auc']/y_te.mean():.1f}×** kazanç."))

,low,median,high
roc_auc,0.621,0.715,0.805
pr_auc,0.106,0.174,0.273
precision,0.097,0.164,0.240
recall,0.421,0.600,0.778


Baseline (dummy) test ROC-AUC = 0.50, PR-AUC = 0.083 (taban oran). Model PR-AUC **0.164** → taban orana göre ≈ **2.0×** kazanç.

**Okuma:** Aralıklar **geniş**. ROC-AUC için aralık 0,5'in belirgin üstünde → model gerçekten bir şey öğreniyor. Ama PR-AUC ve precision aralıkları geniş: *"test'te 0,16 çıktı"* ile *"CV'de 0,27 çıktı"* arasındaki fark bu belirsizlik içinde. Tek bir sayıya fazla güvenmeyin.

## 8.5 İş açısından: Kümülatif kazanç (Gain) ve Lift
"Müşterileri risk skoruna göre sıralayıp en riskli %10'unu ararsak ayrılanların yüzde kaçını yakalarız?"

In [8]:
gain = E.gain_table(y_te, p_te); display(gain.round(3))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([0] + list(gain.top_pct), [0] + list(gain.capture_rate * 100), marker="o", label="Model"); ax[0].plot([0, 100], [0, 100], "k--", label="Rastgele"); ax[0].legend()
ax[0].set_xlabel("Aranan müşteri (%)"); ax[0].set_ylabel("Yakalanan ayrılan (%)"); ax[0].set_title("Kümülatif kazanç (test)")
ax[1].bar(gain.top_pct.astype(str), gain.lift, color="#4c78a8"); ax[1].axhline(1, color="k", ls="--"); ax[1].set_xlabel("En riskli ilk %"); ax[1].set_ylabel("Lift"); ax[1].set_title("Kümülatif lift")
plt.tight_layout(); plt.show()

,top_pct,customers,churners_caught,capture_rate,lift
0,10,36,7,0.233,2.333
1,20,72,13,0.433,2.167
2,30,108,18,0.600,2.000
3,40,144,22,0.733,1.833
4,50,180,25,0.833,1.667
5,60,216,26,0.867,1.444
6,70,252,27,0.900,1.286
7,80,288,28,0.933,1.167
8,90,324,29,0.967,1.074
9,100,360,30,1.000,1.000


**Okuma:** Lift ≈ 2,3 demek: en riskli %10'luk dilim, rastgele seçime göre **yaklaşık 2,3 kat** fazla ayrılan içeriyor (n küçük: 36 müşteri, 7 ayrılan). Model mucize değil ama **sıralamayı rastgeleden belirgin şekilde iyileştiriyor.**

## 8.6 Hata analizi: kimleri kaçırıyoruz?

In [9]:
te = test.assign(score=p_te, pred=pred)
fn = te[(te.churn == 1) & (te.pred == 0)]; tp = te[(te.churn == 1) & (te.pred == 1)]
cols = ["tenure_months", "satisfaction_score", "complaints_6m", "support_calls_6m", "late_payments_6m"]
cmp = pd.DataFrame({"Yakalanan (TP)": tp[cols].mean(), "Kaçırılan (FN)": fn[cols].mean(), "Kalan müşteriler": te[te.churn == 0][cols].mean()})
display(cmp); display(pd.crosstab(te.contract_type, [te.churn, te.pred], rownames=["sözleşme"], colnames=["churn", "tahmin"]))

,Yakalanan (TP),Kaçırılan (FN),Kalan müşteriler
tenure_months,19.444,39.333,37.118
satisfaction_score,5.840,6.492,6.299
complaints_6m,0.722,0.333,0.397
support_calls_6m,1.833,1.250,1.252
late_payments_6m,0.444,0.583,0.567


churn      0      1    
tahmin     0   1  0   1
sözleşme               
Annual    96   8  3   0
Monthly   92  84  7  18
Two-Year  50   0  2   0

**Okuma:** Kaçırılan 12 ayrılanın ortalama abonelik süresi (~39 ay) yakalananlardan (~19 ay) çok daha uzun, şikâyetleri daha az ve memnuniyetleri daha yüksek; 5'i yıllık/iki yıllık sözleşmeli. Yani model, "klasik riskli profil"e uymayan ayrılanları kaçırıyor. Bu veride gözlenmeyen nedenler (rakip teklifi, taşınma vb.) olabilir. Dikkat: n = 12, bu bir hipotezdir, kanıt değil.

## 8.7 Hangi değişkenler önemli?
İki bakış: **(a) lojistik regresyon katsayıları**, **(b) çapraz doğrulamalı permütasyon önemi** (bir sütunu karıştırınca PR-AUC ne kadar düşüyor?). Küçük testte permütasyon önemi çok gürültülü olduğu için eğitim verisinde CV ile hesaplanır.

In [10]:
coef = pd.read_csv(C.ROOT / "reports" / "model_coefficients.csv"); perm = pd.read_csv(C.ROOT / "reports" / "permutation_importance.csv")
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
c = coef.sort_values("coef"); ax[0].barh(c.feature, c.coef, color=np.where(c.coef > 0, "#e45756", "#54a24b")); ax[0].axvline(0, color="k"); ax[0].set_title("Katsayılar (+ riski artırır)")
p = perm.sort_values("importance"); ax[1].barh(p.feature, p.importance, xerr=p["std"], color="#4c78a8"); ax[1].set_title("Permütasyon önemi (train CV)")
plt.tight_layout(); plt.show()
display(coef.round(3))

,feature,coef,odds_ratio
0,contract_type_Monthly,0.947,2.578
1,complaints_6m,0.248,1.281
2,support_calls_6m,0.168,1.183
3,contract_type_Annual,-0.107,0.898
4,satisfaction_score,-0.460,0.631
5,tenure_months,-0.532,0.588
6,contract_type_Two-Year,-0.859,0.423


**Okuma:** Risk artıranlar: aylık sözleşme, şikâyet, destek araması. Risk azaltanlar: uzun abonelik süresi, yüksek memnuniyet, iki yıllık sözleşme (sayısal değişkenler standartlaştırıldığı için katsayılar "1 standart sapma" etkisidir). Permütasyon önemi: memnuniyet, sözleşme ve süre en önde; şikâyet daha küçük; `support_calls_6m` ≈ 0 (katkısı belirsiz). Std değerleri büyük olduğundan sıralamaya fazla güvenme. (Önem ≠ nedensellik.)

## 8.8 Modeli kaydetme ve yeniden yükleme
`joblib.dump(pipeline)` — **tüm Pipeline** (özellik üretimi + imputer + scaler + encoder + model) tek dosyada kaydedilir. Böylece API/Streamlit ham veriyi doğrudan verir; dönüşümleri yeniden yazmak gerekmez.

In [11]:
loaded = joblib.load(C.MODEL_PATH)
new_customer = pd.DataFrame([{"tenure_months": 4, "satisfaction_score": 3.5, "complaints_6m": 1, "support_calls_6m": 4,
                              "late_payments_6m": 2, "contract_type": "Monthly", "plan": "Basic"}])
risk = loaded.predict_proba(new_customer)[:, 1][0]
display(Markdown(f"Yeni müşteri (4 aylık, düşük memnuniyet, aylık sözleşme, 4 destek araması) risk skoru: **{risk:.1%}** → kademe: **{E.risk_tier(risk, thr)}**  (taban oran ≈ %8; eşik {thr:.0%})"))
assert np.allclose(loaded.predict_proba(X_te)[:, 1], p_te)
print("✔ Yüklenen model, kayıtlı modelle aynı tahminleri üretiyor.")

Yeni müşteri (4 aylık, düşük memnuniyet, aylık sözleşme, 4 destek araması) risk skoru: **47.9%** → kademe: **High**  (taban oran ≈ %8; eşik 10%)

✔ Yüklenen model, kayıtlı modelle aynı tahminleri üretiyor.


## 8.9 Sonuç özeti
| | |
|---|---|
| Model | Lojistik Regresyon, 5 girdi |
| CV PR-AUC / ROC-AUC (train) | ≈ 0,27 / 0,75 |
| Test ROC-AUC / PR-AUC | ≈ 0,71 / 0,16 (geniş güven aralığı) |
| Eşik (50 TL maliyet varsayımı) | 0,10 → müşterilerin ≈ %28-31'i |
| Test'te yakalama / precision | ≈ %60 / ≈ %16 |
| Lift (en riskli %10) | ≈ 2,3× |

**Bu modeli savunmanın dürüst yolu:** "Zayıf-orta güçlü bir sinyal var; model rastgeleden belirgin şekilde iyi, düşük maliyetli kampanyalarda değer üretiyor, pahalı kampanyalarda yetersiz. Gerçek maliyetler ve bir A/B testi olmadan canlıya alınmamalı."